# 03 — Restaurant Survival Modeling: Time-to-Closure Analysis

Cox Proportional Hazards on official license data to estimate restaurant closure risk.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## Synthetic Restaurant History


In [ ]:
from src.models.survival_model import (
    build_synthetic_restaurant_history,
    SurvivalModelBundle,
)

df = build_synthetic_restaurant_history(n=300, seed=42)
print("Shape:", df.shape)
print(df.describe().round(2))
print(f"\nEvent (closure) rate: {df['event_observed'].mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df["duration_days"], bins=30, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Duration (days)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Restaurant Lifespans")
plt.tight_layout()
plt.show()

## Kaplan-Meier Survival Curves


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, mask, color in [
    ("Low rent pressure", df["rent_pressure"] < 0.4, "#55A868"),
    ("High rent pressure", df["rent_pressure"] >= 0.4, "#C44E52"),
]:
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, "duration_days"], df.loc[mask, "event_observed"], label=label)
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

ax.set_xlabel("Days open")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier Survival — Rent Pressure Stratification")
ax.legend()
plt.tight_layout()
plt.show()

## Cox PH Model


In [ ]:
model = SurvivalModelBundle(baseline="cox")
model.fit(df)
print(f"Fitted: {model.fitted_}")
print(f"Uses heuristic: {model.uses_heuristic_}")

if model.cox_model_ is not None:
    model.cox_model_.print_summary()

## Risk Prediction on Candidate Zones


In [ ]:
candidate_features = pd.DataFrame(
    {
        "rent_pressure": [
            0.2,
            0.4,
            0.6,
            0.8,
            0.3,
            0.5,
            0.7,
            0.1,
            0.9,
            0.35,
            0.25,
            0.45,
            0.65,
            0.15,
            0.55,
            0.75,
            0.05,
            0.85,
            0.3,
            0.6,
        ],
        "competition_score": [
            0.2,
            0.3,
            0.5,
            0.7,
            0.25,
            0.45,
            0.65,
            0.15,
            0.8,
            0.3,
            0.2,
            0.4,
            0.6,
            0.1,
            0.5,
            0.7,
            0.05,
            0.75,
            0.35,
            0.55,
        ],
        "inspection_grade_numeric": [
            1.1,
            1.3,
            1.5,
            2.0,
            1.2,
            1.4,
            1.8,
            1.0,
            2.5,
            1.3,
            1.1,
            1.4,
            1.7,
            1.0,
            1.5,
            1.9,
            1.0,
            2.2,
            1.2,
            1.6,
        ],
    }
)
risk = model.predict_risk(candidate_features)
print(risk.describe())

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(risk)), risk.sort_values(), color="#DD8452", edgecolor="white")
ax.set_xlabel("Candidate zone (sorted by risk)")
ax.set_ylabel("Closure risk [0–1]")
ax.set_title("Predicted Closure Risk per Candidate Zone")
plt.tight_layout()
plt.show()

## Concordance Index


In [ ]:
from lifelines.utils import concordance_index

if model.cox_model_ is not None:
    risk_train = model.predict_risk(df)
    c_index = concordance_index(df["duration_days"], -risk_train, df["event_observed"])
    print(f"Concordance index: {c_index:.3f}")
else:
    print("Heuristic model — concordance index not applicable")

## Summary

- Higher rent pressure and competition score both increase closure risk.
- Inspection grade is the strongest single predictor.
- Zones with closure risk < 0.35 are commercially viable targets.
- Next: integrate risk scores as `survival_score = 1 - closure_risk` in the CMF opening score.
